# TCN 从零实现：冷却设备升温预警

## 面试问题

面试时我会先定义因果卷积：时刻 t 的输出只能读取 t 及以前的输入，不能通过对称 padding 偷看未来。TCN 用扩张卷积在不大幅增加层数的情况下扩大感受野，并用残差连接稳定梯度。时间序列切分必须按时间而不是随机打散，特征缩放也只能用训练窗口统计量。基线应先用最后温度或斜率规则，确认网络是否真的利用了形状信息。中间层激活、感受野和逐设备概率都应可观察。下面用 12 台冷却设备的八分钟窗口手写因果卷积与 TCN，并复现未来泄漏。

## 真实案例

每条样本是某台冷却设备连续八分钟的温度，标签表示接下来两分钟是否会越过告警线。数据包含持续上升、稳定高温、下降和振荡等真实形状；为了教学离线构造且直接在小样本上拟合，省略了季节、负载与传感器缺失。

本实验是为了看清机制而构造的离线小样本，不代表线上收益，也不能外推到开放分布。

In [1]:
import torch  # 导入 PyTorch 以实现因果卷积和真实反向传播。
from torch import nn  # 导入神经网络基础层。
import torch.nn.functional as F  # 导入左侧填充、激活和损失函数。
torch.manual_seed(29)  # 固定随机种子以保证实验可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小模型运行。
device_ids = [f"冷机-{index:02d}" for index in range(1, 13)]  # 创建十二台脱敏设备编号。
raw_sequences = torch.tensor([[54, 56, 58, 60, 63, 66, 69, 72], [60, 61, 63, 65, 67, 69, 71, 73], [50, 52, 55, 58, 62, 66, 70, 74], [68, 68, 69, 70, 71, 72, 73, 74], [45, 47, 50, 54, 59, 65, 72, 78], [62, 63, 64, 66, 68, 70, 73, 76], [76, 75, 74, 73, 72, 71, 70, 69], [74, 74, 74, 74, 74, 74, 74, 74], [80, 78, 76, 74, 72, 70, 68, 66], [70, 71, 70, 71, 70, 71, 70, 71], [73, 73, 72, 72, 71, 71, 70, 70], [75, 75, 75, 75, 75, 75, 75, 75]], dtype=torch.float32)  # 保存八分钟温度窗口。
labels = torch.tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=torch.long)  # 保存未来两分钟是否告警的标签。
normalized_sequences = ((raw_sequences - 65.0) / 10.0).unsqueeze(1)  # 用固定训练统计量缩放并增加通道维。
print("设备      八分钟温度序列                         未来告警")  # 打印输入预览表头。
for index, device_id in enumerate(device_ids):  # 逐设备展示有业务语义的原始时间序列。
    temperatures = ",".join(str(int(value)) for value in raw_sequences[index].tolist())  # 把温度张量转成可读字符串。
    print(f"{device_id}  [{temperatures:<23}]  {labels[index].item()}")  # 输出当前设备窗口和监督标签。
print(f"输入张量形状={tuple(normalized_sequences.shape)}，含义=(设备, 温度通道, 分钟)")  # 解释网络实际接收的三维张量。

设备      八分钟温度序列                         未来告警
冷机-01  [54,56,58,60,63,66,69,72]  1
冷机-02  [60,61,63,65,67,69,71,73]  1
冷机-03  [50,52,55,58,62,66,70,74]  1
冷机-04  [68,68,69,70,71,72,73,74]  1
冷机-05  [45,47,50,54,59,65,72,78]  1
冷机-06  [62,63,64,66,68,70,73,76]  1
冷机-07  [76,75,74,73,72,71,70,69]  0
冷机-08  [74,74,74,74,74,74,74,74]  0
冷机-09  [80,78,76,74,72,70,68,66]  0
冷机-10  [70,71,70,71,70,71,70,71]  0
冷机-11  [73,73,72,72,71,71,70,70]  0
冷机-12  [75,75,75,75,75,75,75,75]  0
输入张量形状=(12, 1, 8)，含义=(设备, 温度通道, 分钟)


## 基线：只看最后一分钟温度

简单规则把末值不低于 72℃ 判为未来告警。它会把稳定在 74–75℃ 但没有继续上升的设备误报，因此为 TCN 是否学习到趋势形状提供同数据对照。

In [2]:
baseline_predictions = (raw_sequences[:, -1] >= 72.0).long()  # 根据最后一分钟温度产生阈值预测。
baseline_accuracy = (baseline_predictions == labels).float().mean().item()  # 计算十二台设备上的基线准确率。
print("设备      末值  基线预测  真实标签")  # 打印逐设备基线结果表头。
for index, device_id in enumerate(device_ids):  # 逐设备观察末值规则的误报和漏报。
    print(f"{device_id}  {raw_sequences[index, -1]:.0f}      {baseline_predictions[index].item()}         {labels[index].item()}")  # 输出末值、规则预测和真实标签。
print(f"末值阈值基线准确率={baseline_accuracy:.1%}")  # 汇总同一数据上的基线指标。

设备      末值  基线预测  真实标签
冷机-01  72      1         1
冷机-02  73      1         1
冷机-03  74      1         1
冷机-04  74      1         1
冷机-05  78      1         1
冷机-06  76      1         1
冷机-07  69      0         0
冷机-08  74      1         0
冷机-09  66      0         0
冷机-10  71      0         0
冷机-11  70      0         0
冷机-12  75      1         0
末值阈值基线准确率=83.3%


## 手写核心：左填充、扩张卷积与残差

`CausalConv1d` 明确只在左侧补零，再调用基础卷积；两个残差块的 dilation 分别为 1 和 2，卷积核宽度为 3，因此最后时刻感受野是 `1 + 2×1 + 2×2 = 7` 分钟。

In [3]:
class CausalConv1d(nn.Module):  # 定义只读取当前及历史输入的一维卷积层。
    def __init__(self, input_channels, output_channels, kernel_size, dilation):  # 接收通道数、卷积核和扩张率。
        super().__init__()  # 初始化父类以注册卷积参数。
        self.left_padding = (kernel_size - 1) * dilation  # 计算保持长度所需的纯左侧填充量。
        self.convolution = nn.Conv1d(input_channels, output_channels, kernel_size, dilation=dilation)  # 创建不含自动 padding 的基础卷积。
    def forward(self, sequence):  # 对输入序列执行因果卷积。
        padded = F.pad(sequence, (self.left_padding, 0))  # 只在时间轴左侧补零以阻断未来信息。
        return self.convolution(padded)  # 对左填充序列执行扩张卷积并保持原长度。
class ResidualTemporalBlock(nn.Module):  # 定义一个因果卷积残差块。
    def __init__(self, channels, dilation):  # 根据通道数和扩张率创建残差块。
        super().__init__()  # 初始化父类以注册子层。
        self.causal_convolution = CausalConv1d(channels, channels, 3, dilation)  # 创建卷积核为三的因果扩张卷积。
        self.normalization = nn.LayerNorm(channels)  # 在每个时间步的通道维执行归一化。
    def forward(self, sequence):  # 计算卷积分支并与原输入相加。
        convolved = self.causal_convolution(sequence)  # 提取当前扩张尺度的历史模式。
        normalized = self.normalization(convolved.transpose(1, 2)).transpose(1, 2)  # 调整维度后执行通道归一化。
        return torch.relu(sequence + normalized)  # 加入残差连接并应用 ReLU。
class TinyTCN(nn.Module):  # 定义用于设备告警的两层扩张 TCN。
    def __init__(self):  # 创建输入投影、两个残差块和分类头。
        super().__init__()  # 初始化父类以注册所有子模块。
        self.input_projection = nn.Conv1d(1, 8, 1)  # 把单温度通道投影成八维表示。
        self.block_one = ResidualTemporalBlock(8, dilation=1)  # 用 dilation=1 捕获局部趋势。
        self.block_two = ResidualTemporalBlock(8, dilation=2)  # 用 dilation=2 扩大历史感受野。
        self.classifier = nn.Linear(8, 2)  # 根据最后时间步表示预测是否告警。
    def forward(self, sequence):  # 完成一次 TCN 前向传播并暴露中间激活。
        projected = torch.relu(self.input_projection(sequence))  # 把标准化温度映射到隐藏通道。
        first_hidden = self.block_one(projected)  # 计算第一尺度的因果卷积表示。
        second_hidden = self.block_two(first_hidden)  # 计算更大感受野的卷积表示。
        logits = self.classifier(second_hidden[:, :, -1])  # 使用最后时刻隐藏表示输出二分类 logits。
        return logits, first_hidden, second_hidden  # 返回预测和两个卷积块的中间激活。
tcn = TinyTCN()  # 实例化手写因果 TCN。
receptive_field = 1 + 2 * 1 + 2 * 2  # 按卷积核和扩张率计算最后输出的历史感受野。
print(tcn)  # 展示手写 TCN 的实际模块结构。
print(f"理论感受野={receptive_field} 分钟，可训练参数={sum(parameter.numel() for parameter in tcn.parameters())}")  # 输出网络覆盖范围与参数规模。

TinyTCN(
  (input_projection): Conv1d(1, 8, kernel_size=(1,), stride=(1,))
  (block_one): ResidualTemporalBlock(
    (causal_convolution): CausalConv1d(
      (convolution): Conv1d(8, 8, kernel_size=(3,), stride=(1,))
    )
    (normalization): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  )
  (block_two): ResidualTemporalBlock(
    (causal_convolution): CausalConv1d(
      (convolution): Conv1d(8, 8, kernel_size=(3,), stride=(1,), dilation=(2,))
    )
    (normalization): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  )
  (classifier): Linear(in_features=8, out_features=2, bias=True)
)
理论感受野=7 分钟，可训练参数=466


In [4]:
optimizer = torch.optim.Adam(tcn.parameters(), lr=0.02)  # 创建优化器更新卷积核和分类头。
loss_trace = []  # 保存每轮训练损失用于检查优化过程。
first_gradient_norm = 0.0  # 预留首轮扩张卷积梯度范数。
for epoch in range(301):  # 在十二条教学窗口上执行三百零一次更新。
    optimizer.zero_grad()  # 清除上一轮累积梯度。
    logits, first_hidden, second_hidden = tcn(normalized_sequences)  # 运行手写 TCN 前向传播。
    loss = F.cross_entropy(logits, labels)  # 计算未来告警分类交叉熵。
    loss.backward()  # 反向传播到两个因果卷积块。
    if epoch == 0:  # 首轮记录实际卷积核梯度。
        first_gradient_norm = tcn.block_one.causal_convolution.convolution.weight.grad.norm().item()  # 读取第一扩张卷积的梯度范数。
    optimizer.step()  # 根据梯度更新全部模型参数。
    loss_trace.append(loss.item())  # 保存当前轮损失值。
    if epoch in [0, 25, 100, 300]:  # 选择少数关键轮次输出训练轨迹。
        current_accuracy = (logits.argmax(dim=1) == labels).float().mean().item()  # 计算当前训练准确率。
        print(f"epoch={epoch:03d} loss={loss.item():.4f} accuracy={current_accuracy:.1%}")  # 输出损失和准确率变化。
tcn.eval()  # 切换到评估模式获得稳定逐样本结果。
with torch.no_grad():  # 关闭评估阶段的梯度记录。
    tcn_logits, first_activations, second_activations = tcn(normalized_sequences)  # 计算最终预测与两层激活。
tcn_probabilities = torch.softmax(tcn_logits, dim=1)[:, 1]  # 提取未来告警类别概率。
tcn_predictions = tcn_logits.argmax(dim=1)  # 选择概率最大的告警类别。
tcn_accuracy = (tcn_predictions == labels).float().mean().item()  # 计算 TCN 在同一数据上的准确率。
print(f"首轮卷积核梯度范数={first_gradient_norm:.6f}")  # 输出非零梯度证明真实反向传播发生。
print(f"冷机-01 第一块最后时刻激活={[round(value, 3) for value in first_activations[0, :, -1].tolist()]}")  # 展示局部趋势卷积后的中间表示。
print(f"冷机-01 第二块最后时刻激活={[round(value, 3) for value in second_activations[0, :, -1].tolist()]}")  # 展示扩大感受野后的中间表示。

epoch=000 loss=0.7283 accuracy=50.0%


epoch=025 loss=0.0060 accuracy=100.0%


epoch=100 loss=0.0002 accuracy=100.0%


epoch=300 loss=0.0001 accuracy=100.0%
首轮卷积核梯度范数=1.636390
冷机-01 第一块最后时刻激活=[1.585, 0.0, 1.151, 1.74, 0.718, 1.289, 0.0, 0.435]
冷机-01 第二块最后时刻激活=[1.091, 2.447, 1.13, 4.569, 0.0, 0.0, 0.0, 1.22]


## 结果解读

稳定高温与持续升温的末值可能相同，TCN 的价值在于读取最近七分钟的形状。逐设备结果能直接看出稳定在 74℃、75℃ 的误报是否被纠正。

In [5]:
print("设备      末值  真实  基线  TCN概率  TCN预测")  # 打印同数据逐设备对照表头。
for index, device_id in enumerate(device_ids):  # 遍历全部设备展示预测和概率。
    print(f"{device_id}  {raw_sequences[index, -1]:.0f}    {labels[index].item()}     {baseline_predictions[index].item()}     {tcn_probabilities[index]:.3f}    {tcn_predictions[index].item()}")  # 输出当前设备的完整对照结果。
print(f"同数据准确率：末值基线={baseline_accuracy:.1%}，手写 TCN={tcn_accuracy:.1%}")  # 汇总两个方法的同口径表现。

设备      末值  真实  基线  TCN概率  TCN预测
冷机-01  72    1     1     1.000    1
冷机-02  73    1     1     1.000    1
冷机-03  74    1     1     1.000    1
冷机-04  74    1     1     1.000    1
冷机-05  78    1     1     1.000    1
冷机-06  76    1     1     1.000    1
冷机-07  69    0     0     0.000    0
冷机-08  74    0     1     0.000    0
冷机-09  66    0     0     0.000    0
冷机-10  71    0     0     0.000    0
冷机-11  70    0     0     0.000    0
冷机-12  75    0     1     0.000    0
同数据准确率：末值基线=83.3%，手写 TCN=100.0%


## 失败案例：对称 padding 偷看下一分钟

下面把卷积核固定为全 1，并只修改第 6 分钟的未来值。对称 padding 会让第 5 分钟输出随未来变化；纯左填充的因果卷积在第 5 分钟保持不变。

In [6]:
symmetric_convolution = nn.Conv1d(1, 1, 3, padding=1, bias=False)  # 创建会同时读取左右邻居的错误卷积。
causal_convolution = CausalConv1d(1, 1, 3, dilation=1)  # 创建只读取当前及过去的正确卷积。
with torch.no_grad():  # 关闭梯度并设置完全可解释的固定卷积核。
    symmetric_convolution.weight.fill_(1.0)  # 把对称卷积核三个权重都设为一。
    causal_convolution.convolution.weight.fill_(1.0)  # 把因果卷积核三个权重都设为一。
    causal_convolution.convolution.bias.zero_()  # 清零因果卷积偏置避免干扰对照。
prefix_original = torch.zeros(1, 1, 8)  # 创建全零的原始八分钟序列。
prefix_future_changed = prefix_original.clone()  # 复制序列用于只改变未来时刻。
prefix_future_changed[0, 0, 5] = 10.0  # 只修改索引五的未来值而保持前缀不变。
symmetric_before = symmetric_convolution(prefix_original)[0, 0, 4].item()  # 计算错误卷积在索引四的原输出。
symmetric_after = symmetric_convolution(prefix_future_changed)[0, 0, 4].item()  # 计算未来改变后的错误卷积输出。
causal_before = causal_convolution(prefix_original)[0, 0, 4].item()  # 计算因果卷积在索引四的原输出。
causal_after = causal_convolution(prefix_future_changed)[0, 0, 4].item()  # 计算未来改变后的因果卷积输出。
print(f"对称 padding：未来修改前={symmetric_before:.1f}，修改后={symmetric_after:.1f}")  # 展示错误实现受到未来值污染。
print(f"纯左 padding：未来修改前={causal_before:.1f}，修改后={causal_after:.1f}")  # 展示正确实现保持前缀不变。
print("修复结论：预测时刻之前只能做左填充，验证切分也必须尊重时间。")  # 给出防止时间泄漏的工程结论。

对称 padding：未来修改前=0.0，修改后=10.0
纯左 padding：未来修改前=0.0，修改后=0.0
修复结论：预测时刻之前只能做左填充，验证切分也必须尊重时间。


## 生产差距

真实设备流需要按设备和时间划分训练/验证、只用训练统计量归一化，并处理缺测、乱序、采样频率漂移和告警延迟。模型还需滚动回测、概率校准、推理延迟监控和规则兜底。本实验在全部小样本上拟合，只证明 TCN 机制，不证明跨设备泛化。

## 最小回归测试

In [7]:
assert len(device_ids) >= 6  # 保证时间序列案例包含足够多的真实设备样本。
assert receptive_field == 7  # 保证两层扩张卷积的感受野计算正确。
assert loss_trace[-1] < loss_trace[0]  # 保证真实训练使交叉熵损失下降。
assert first_gradient_norm > 0.0  # 保证因果卷积核获得了非零梯度。
assert tcn_accuracy > baseline_accuracy  # 保证 TCN 在同一数据上纠正了末值规则错误。
assert symmetric_after != symmetric_before  # 保证错误对称 padding 确实读取了未来值。
assert causal_after == causal_before  # 保证纯左填充阻断了未来信息。
print("回归测试通过：TCN 训练、感受野和因果性均符合预期。")  # 输出集中断言的最终验收结果。

回归测试通过：TCN 训练、感受野和因果性均符合预期。
